In [0]:
%run /Workspace/Users/fayelatyr61@gmail.com/azure-databricks-realtime-health-platform/01-config

In [0]:
# Databricks notebook source

# ============================================================
# PRODUCER
# ============================================================
# Cette classe sert à préparer automatiquement les données
# utilisées pour les tests d'intégration.
#
# Principe :
#
# test_data
#    ↓
# Producer
#    ↓
# landing zone / raw
#    ↓
# Bronze → Silver → Gold
#
# On dispose de deux jeux de données :
# Payload 1 et Payload 2.
# ============================================================

class Producer:

    # --------------------------------------------------------
    # INITIALISATION
    # --------------------------------------------------------
    # Le constructeur récupère les chemins définis dans Config.
    #
    # test_data_dir :
    # contient les fichiers de test préparés à l'avance.
    #
    # landing_zone :
    # correspond à la zone /raw surveillée par la couche Bronze.
    # --------------------------------------------------------

    def __init__(self):

        self.conf = Config()

        self.landing_zone = (
            self.conf.base_dir_data
            + "/raw"
        )

        self.test_data_dir = (
            self.conf.base_dir_data
            + "/test_data"
        )


    # --------------------------------------------------------
    # USER REGISTRATION
    # --------------------------------------------------------
    # Copie les données d'inscription des utilisateurs
    # depuis test_data vers la landing zone.
    #
    # Exemple :
    #
    # 1-registered_users_1.csv
    #       ↓
    # raw/registered_users_bz/
    #
    # set_num = 1 → Payload 1
    # set_num = 2 → Payload 2
    # --------------------------------------------------------

    def user_registration(self, set_num):

        source = (
            f"{self.test_data_dir}/"
            f"1-registered_users_{set_num}.csv"
        )

        target = (
            f"{self.landing_zone}/"
            f"registered_users_bz/"
            f"1-registered_users_{set_num}.csv"
        )

        print(
            f"Producing {source}...",
            end=""
        )

        dbutils.fs.cp(
            source,
            target
        )

        print("Done")


    # --------------------------------------------------------
    # USER PROFILE CDC
    # --------------------------------------------------------
    # Copie les événements CDC du profil utilisateur.
    #
    # Ces fichiers contiennent les événements :
    # new / update
    #
    # Ils sont envoyés vers kafka_multiplex_bz car ils
    # simulent des événements provenant de Kafka.
    #
    # 2-user_info_X.json
    #       ↓
    # raw/kafka_multiplex_bz/
    # --------------------------------------------------------

    def profile_cdc(self, set_num):

        source = (
            f"{self.test_data_dir}/"
            f"2-user_info_{set_num}.json"
        )

        target = (
            f"{self.landing_zone}/"
            f"kafka_multiplex_bz/"
            f"2-user_info_{set_num}.json"
        )

        print(
            f"Producing {source}...",
            end=""
        )

        dbutils.fs.cp(
            source,
            target
        )

        print("Done")


    # --------------------------------------------------------
    # WORKOUT
    # --------------------------------------------------------
    # Copie les événements de workout.
    #
    # Ces fichiers contiennent notamment :
    # start / stop
    #
    # Ils seront utilisés plus tard par Silver pour reconstruire
    # les séances complètes.
    #
    # 4-workout_X.json
    #       ↓
    # raw/kafka_multiplex_bz/
    # --------------------------------------------------------

    def workout(self, set_num):

        source = (
            f"{self.test_data_dir}/"
            f"4-workout_{set_num}.json"
        )

        target = (
            f"{self.landing_zone}/"
            f"kafka_multiplex_bz/"
            f"4-workout_{set_num}.json"
        )

        print(
            f"Producing {source}...",
            end=""
        )

        dbutils.fs.cp(
            source,
            target
        )

        print("Done")


    # --------------------------------------------------------
    # BPM
    # --------------------------------------------------------
    # Copie les mesures de fréquence cardiaque.
    #
    # BPM = Beats Per Minute
    #
    # Ces fichiers représentent le plus gros volume de données
    # du test.
    #
    # 3-bpm_X.json
    #       ↓
    # raw/kafka_multiplex_bz/
    # --------------------------------------------------------

    def bpm(self, set_num):

        source = (
            f"{self.test_data_dir}/"
            f"3-bpm_{set_num}.json"
        )

        target = (
            f"{self.landing_zone}/"
            f"kafka_multiplex_bz/"
            f"3-bpm_{set_num}.json"
        )

        print(
            f"Producing {source}...",
            end=""
        )

        dbutils.fs.cp(
            source,
            target
        )

        print("Done")


    # --------------------------------------------------------
    # GYM LOGINS
    # --------------------------------------------------------
    # Copie les événements d'entrée et de sortie des salles.
    #
    # 5-gym_logins_X.csv
    #       ↓
    # raw/gym_logins_bz/
    #
    # Ces données seront ensuite transformées en gym_logs
    # dans la couche Silver.
    # --------------------------------------------------------

    def gym_logins(self, set_num):

        source = (
            f"{self.test_data_dir}/"
            f"5-gym_logins_{set_num}.csv"
        )

        target = (
            f"{self.landing_zone}/"
            f"gym_logins_bz/"
            f"5-gym_logins_{set_num}.csv"
        )

        print(
            f"Producing {source}...",
            end=""
        )

        dbutils.fs.cp(
            source,
            target
        )

        print("Done")


    # --------------------------------------------------------
    # PRODUCE
    # --------------------------------------------------------
    # Fonction principale du Producer.
    #
    # Elle permet de produire tout un payload en une commande.
    #
    # producer.produce(1)
    # → copie le Payload 1
    #
    # producer.produce(2)
    # → copie le Payload 2
    #
    # Cela simule l'arrivée de nouvelles données dans
    # la landing zone.
    # --------------------------------------------------------

    def produce(self, set_num):

        import time

        start = int(time.time())

        print(
            f"\nProducing test data set {set_num}..."
        )

        if set_num <= 2:

            self.user_registration(
                set_num
            )

            self.profile_cdc(
                set_num
            )

            self.workout(
                set_num
            )

            self.gym_logins(
                set_num
            )

        if set_num <= 10:

            self.bpm(
                set_num
            )

        print(
            f"Test data set {set_num} produced in "
            f"{int(time.time()) - start} seconds"
        )


    # --------------------------------------------------------
    # VALIDATE COUNT
    # --------------------------------------------------------
    # Fonction utilitaire qui vérifie que les fichiers ont été
    # correctement copiés dans la landing zone.
    #
    # Elle :
    # 1. lit les fichiers concernés
    # 2. compte les lignes
    # 3. compare avec le nombre attendu
    #
    # Cela permet de détecter un problème avant de lancer
    # Bronze → Silver → Gold.
    # --------------------------------------------------------

    def _validate_count(
        self,
        file_format,
        location,
        expected_count
    ):

        print(
            f"Validating {location}...",
            end=""
        )

        target = (
            f"{self.landing_zone}/"
            f"{location}_*.{file_format}"
        )

        actual_count = (
            spark.read
            .format(file_format)
            .option(
                "header",
                "true"
            )
            .load(target)
            .count()
        )

        assert actual_count == expected_count, (
            f"Expected {expected_count:,} records, "
            f"found {actual_count:,} "
            f"in {location}"
        )

        print(
            f"Found {actual_count:,} / "
            f"Expected {expected_count:,} "
            f"records: Success"
        )


    # --------------------------------------------------------
    # VALIDATE
    # --------------------------------------------------------
    # Vérifie les cinq types de données produits.
    #
    # Pour Payload 1, les comptes attendus sont par exemple :
    #
    # registered_users = 5
    # user_info = 7
    # bpm = 253801
    # workout = 16
    # gym_logins = 8
    #
    # Avec Payload 2, les données s'accumulent car on teste
    # le comportement incrémental du pipeline.
    # --------------------------------------------------------

    def validate(self, sets):

        import time

        start = int(time.time())

        print(
            f"\nValidating test data "
            f"{sets} sets..."
        )

        self._validate_count(
            "csv",
            "registered_users_bz/1-registered_users",
            5 if sets == 1 else 10
        )

        self._validate_count(
            "json",
            "kafka_multiplex_bz/2-user_info",
            7 if sets == 1 else 13
        )

        self._validate_count(
            "json",
            "kafka_multiplex_bz/3-bpm",
            sets * 253801
        )

        self._validate_count(
            "json",
            "kafka_multiplex_bz/4-workout",
            16 if sets == 1 else 32
        )

        self._validate_count(
            "csv",
            "gym_logins_bz/5-gym_logins",
            8 if sets == 1 else 16
        )

        print(
            f"Test data validation completed in "
            f"{int(time.time()) - start} seconds"
        )

In [0]:
producer = Producer()

In [0]:
producer.produce(1)
producer.validate(1)

In [0]:
producer.produce(2)
producer.validate(2)